# Feature Engineering & Selection

**Project:** E-commerce Sales Data Analysis & Machine Learning  
**Author:** Muhammad Iqbal Fadel  
**Source:** `scripts/03_model_prep.py`

In [ ]:
%matplotlib inline

import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'Sales_Transaction_v4a_cleaned.csv')
OUT_DIR = os.path.join(ROOT, 'data', 'processed')

print('Loading data')
df = pd.read_csv(DATA_PATH, low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Build customer-level features
cust = df.groupby('CustomerNo').agg(
    Recency=('Date', lambda x: (df['Date'].max() - x.max()).days),
    Frequency=('TransactionNo', 'nunique' if 'TransactionNo' in df.columns else 'count'),
    Monetary=('TotalAmount', 'sum'),
    AvgOrderValue=('TotalAmount', 'mean'),
    AvgItemsPerOrder=('Quantity', lambda x: x.sum() / x.nunique() if x.nunique()>0 else 0),
    DistinctProducts=('ProductNo', 'nunique')
).reset_index()

# Fillna and basic cleaning
cust = cust.fillna(0)

# Create label: Champions (top monetary quartile) as positive class
cust['Label'] = (cust['Monetary'] >= cust['Monetary'].quantile(0.75)).astype(int)

feature_cols = ['Recency','Frequency','Monetary','AvgOrderValue','AvgItemsPerOrder','DistinctProducts']
X = cust[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = cust['Label']

# Feature selection
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)
scores = pd.DataFrame({'feature': feature_cols, 'score': selector.scores_}).sort_values('score', ascending=False)

# Save outputs
cust.to_csv(os.path.join(OUT_DIR, 'model_data_customers.csv'), index=False)
scores.to_csv(os.path.join(OUT_DIR, 'selected_feature_scores.csv'), index=False)
print('Saved model data and feature scores to', OUT_DIR)
print(scores)